# 🍈 DurianVision — Training YOLOv11 di Google Colab

Notebook ini melakukan **Level 2A + 2B + 2C** untuk meningkatkan akurasi deteksi varietas durian:

| Level | Aksi | Dampak |
|:---:|---|:---:|
| **2A** | Model lebih besar (yolo11m/l) | +20-30% |
| **2B** | Balance dataset (augmentasi kelas minoritas) | +15-25% |
| **2C** | Gabungan 2A + 2B + augmentasi agresif | **+35-50%** |

## Persiapan
1. Pastikan Runtime → **Change runtime type → GPU (T4)**
2. Upload file `dataset.zip` (folder `training/dataset/`) ke Google Drive
3. Jalankan semua cell dari atas ke bawah

---

## 📦 Langkah 0: Cek GPU & Install Dependencies

In [ ]:
# Cek GPU tersedia
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_mem / (1024**3)
    print(f"GPU: {gpu_name} ({vram:.1f} GB VRAM)")
else:
    print("⚠️ GPU TIDAK TERSEDIA! Pastikan Runtime → Change runtime type → GPU")
    raise RuntimeError("GPU diperlukan untuk training")

In [ ]:
# Install dependencies
!pip install -q ultralytics albumentations opencv-python-headless pyyaml tqdm
print("\n✅ Dependencies terinstall")

## 📂 Langkah 1: Mount Google Drive & Upload Dataset

### Cara menyiapkan dataset:
1. Di laptop, buka folder `d:\GUI Duren\GUI Duren\training\dataset\`
2. Zip seluruh isi folder tersebut menjadi `dataset.zip` — pastikan struktur di dalam zip:
   ```
   dataset.zip
   ├── data.yaml
   ├── train/
   │   ├── images/
   │   └── labels/
   ├── valid/
   │   ├── images/
   │   └── labels/
   └── test/
       ├── images/
       └── labels/
   ```
3. Upload `dataset.zip` ke **Google Drive** (root folder atau folder manapun)
4. Jalankan cell di bawah

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ==========================================
# ⚙️ KONFIGURASI — UBAH SESUAI KEBUTUHAN
# ==========================================

# Path ke dataset.zip di Google Drive
# Ubah path ini sesuai lokasi file Anda di Drive
DATASET_ZIP_PATH = "/content/drive/MyDrive/dataset.zip"

# Folder kerja di Colab (jangan diubah)
WORK_DIR = "/content/durian_training"
DATASET_DIR = f"{WORK_DIR}/dataset"

# Folder output di Google Drive (hasil training akan disimpan di sini)
OUTPUT_DRIVE = "/content/drive/MyDrive/DurianVision_Training"

print(f"Dataset ZIP : {DATASET_ZIP_PATH}")
print(f"Work Dir    : {WORK_DIR}")
print(f"Output Dir  : {OUTPUT_DRIVE}")

In [ ]:
import os, zipfile, shutil

# Bersihkan folder kerja jika ada
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
os.makedirs(WORK_DIR, exist_ok=True)

# Cek apakah file zip ada
if not os.path.exists(DATASET_ZIP_PATH):
    print(f"❌ File tidak ditemukan: {DATASET_ZIP_PATH}")
    print("\nPastikan:")
    print("1. File dataset.zip sudah diupload ke Google Drive")
    print("2. Path di variabel DATASET_ZIP_PATH sudah benar")
    print("\nContoh path:")
    print('  /content/drive/MyDrive/dataset.zip')
    print('  /content/drive/MyDrive/Kuliah/dataset.zip')
    raise FileNotFoundError(f"Dataset zip tidak ditemukan")

# Extract
print(f"📦 Mengekstrak {DATASET_ZIP_PATH}...")
with zipfile.ZipFile(DATASET_ZIP_PATH, 'r') as z:
    z.extractall(DATASET_DIR)

# Cek apakah ada subfolder dataset/ di dalam zip
data_yaml = os.path.join(DATASET_DIR, 'data.yaml')
if not os.path.exists(data_yaml):
    # Mungkin di subfolder
    for root, dirs, files in os.walk(DATASET_DIR):
        if 'data.yaml' in files:
            # Pindahkan ke DATASET_DIR
            actual_dir = root
            if actual_dir != DATASET_DIR:
                for item in os.listdir(actual_dir):
                    src = os.path.join(actual_dir, item)
                    dst = os.path.join(DATASET_DIR, item)
                    if os.path.exists(dst):
                        if os.path.isdir(dst):
                            shutil.rmtree(dst)
                        else:
                            os.remove(dst)
                    shutil.move(src, dst)
                print(f"  📁 Dataset ditemukan di subfolder: {actual_dir}")
            break

# Verifikasi struktur
required = ['data.yaml', 'train/images', 'train/labels', 'valid/images', 'valid/labels']
for r in required:
    path = os.path.join(DATASET_DIR, r)
    exists = os.path.exists(path)
    print(f"  {'✅' if exists else '❌'} {r}")
    if not exists:
        raise FileNotFoundError(f"Missing: {path}")

print("\n✅ Dataset berhasil diekstrak!")

## 📊 Langkah 2: Analisis Dataset (Sebelum Balancing)

Melihat distribusi kelas untuk memahami seberapa parah imbalance-nya.

In [ ]:
import collections
import yaml
import matplotlib.pyplot as plt
import numpy as np

def analyze_dataset(dataset_dir, title="Dataset"):
    """Analisis distribusi kelas dalam dataset."""
    with open(os.path.join(dataset_dir, 'data.yaml'), 'r') as f:
        config = yaml.safe_load(f)

    names = config.get('names', [])
    nc = config.get('nc', len(names))

    results = {}
    for split in ['train', 'valid', 'test']:
        lbl_dir = os.path.join(dataset_dir, split, 'labels')
        img_dir = os.path.join(dataset_dir, split, 'images')
        if not os.path.exists(lbl_dir):
            continue

        class_counts = collections.Counter()
        img_count = len([f for f in os.listdir(img_dir)
                        if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        total = 0

        for f in os.listdir(lbl_dir):
            if f.endswith('.txt'):
                with open(os.path.join(lbl_dir, f)) as fh:
                    for line in fh:
                        parts = line.strip().split()
                        if parts:
                            class_counts[int(parts[0])] += 1
                            total += 1

        results[split] = {'counts': dict(class_counts), 'total': total, 'images': img_count}

    # Visualisasi
    if 'train' in results:
        counts = results['train']['counts']
        cls_names = [names[i] if i < len(names) else f'class_{i}' for i in sorted(counts.keys())]
        cls_values = [counts[i] for i in sorted(counts.keys())]

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        fig.suptitle(f'📊 {title}', fontsize=14, fontweight='bold')

        # Bar chart
        colors = plt.cm.Set3(np.linspace(0, 1, len(cls_names)))
        bars = ax1.barh(cls_names, cls_values, color=colors)
        ax1.set_xlabel('Jumlah Anotasi')
        ax1.set_title('Distribusi Kelas (Training Set)')
        for bar, val in zip(bars, cls_values):
            ax1.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
                    f'{val}', va='center', fontsize=10)

        # Pie chart
        ax2.pie(cls_values, labels=cls_names, autopct='%1.1f%%', colors=colors)
        ax2.set_title('Proporsi Kelas')

        plt.tight_layout()
        plt.show()

        # Print stats
        print(f"\n{'='*50}")
        print(f"  {title} — Statistik")
        print(f"{'='*50}")
        for split, data in results.items():
            print(f"\n  [{split.upper()}] {data['images']} gambar, {data['total']} anotasi")
            for cls_id in sorted(data['counts']):
                n = data['counts'][cls_id]
                name = names[cls_id] if cls_id < len(names) else f'class_{cls_id}'
                pct = n / data['total'] * 100 if data['total'] > 0 else 0
                print(f"    [{cls_id}] {name:15s}: {n:5d} ({pct:5.1f}%)")

        max_c = max(counts.values())
        min_c = min(counts.values())
        ratio = max_c / min_c if min_c > 0 else float('inf')
        print(f"\n  ⚖️  Imbalance ratio: {ratio:.1f}x")
        if ratio > 3:
            print(f"  ⚠️  Dataset TIDAK SEIMBANG — perlu balancing!")
        else:
            print(f"  ✅ Dataset cukup seimbang")

    return results, names

# Jalankan analisis
original_stats, class_names = analyze_dataset(DATASET_DIR, "Dataset ORIGINAL")

## ⚖️ Langkah 3: Level 2B — Balancing Dataset

Menyeimbangkan dataset dengan **offline augmentation** untuk kelas minoritas:
- Kelas dengan data paling sedikit akan di-augmentasi (flip, rotasi, warna, brightness)
- Target: setiap kelas memiliki jumlah annotation yang setara dengan kelas terbanyak
- Label (bounding box) ikut ditransformasi

In [ ]:
import cv2
import random
import glob
from tqdm import tqdm
import albumentations as A

def balance_dataset(dataset_dir, class_names):
    """
    Level 2B: Balance dataset dengan offline augmentation.
    Menghasilkan gambar + label baru untuk kelas minoritas.
    """
    train_img_dir = os.path.join(dataset_dir, 'train', 'images')
    train_lbl_dir = os.path.join(dataset_dir, 'train', 'labels')

    # Hitung anotasi per kelas DAN kumpulkan file per kelas
    class_counts = collections.Counter()
    class_files = collections.defaultdict(list)  # cls_id -> [(img_path, lbl_path, line_index)]

    for lbl_file in sorted(os.listdir(train_lbl_dir)):
        if not lbl_file.endswith('.txt'):
            continue
        lbl_path = os.path.join(train_lbl_dir, lbl_file)
        # Cari gambar yang sesuai
        base = os.path.splitext(lbl_file)[0]
        img_path = None
        for ext in ['.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG']:
            candidate = os.path.join(train_img_dir, base + ext)
            if os.path.exists(candidate):
                img_path = candidate
                break
        if img_path is None:
            continue

        with open(lbl_path) as f:
            lines = f.readlines()

        file_classes = set()
        for line in lines:
            parts = line.strip().split()
            if parts:
                cls_id = int(parts[0])
                class_counts[cls_id] += 1
                file_classes.add(cls_id)

        # File ini berguna untuk semua kelas yang ada di dalamnya
        for cls_id in file_classes:
            class_files[cls_id].append((img_path, lbl_path))

    if not class_counts:
        print("❌ Tidak ada anotasi ditemukan!")
        return

    max_count = max(class_counts.values())
    target_count = max_count  # Target: semua kelas setara kelas terbanyak

    print(f"\n{'='*50}")
    print(f"  ⚖️  BALANCING DATASET")
    print(f"{'='*50}")
    print(f"\n  Target per kelas: ~{target_count} anotasi")
    print(f"  Kelas terbanyak : {class_names[max(class_counts, key=class_counts.get)]} ({max_count})")

    # Augmentation pipeline (aman untuk bounding box)
    augmenter = A.Compose([
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.7),
        A.HueSaturationValue(hue_shift_limit=15, sat_shift_limit=30, val_shift_limit=25, p=0.7),
        A.GaussNoise(var_limit=(10, 50), p=0.3),
        A.GaussianBlur(blur_limit=(3, 5), p=0.2),
        A.RandomGamma(gamma_limit=(80, 120), p=0.3),
        A.CLAHE(clip_limit=2.0, p=0.2),
        A.Affine(
            rotate=(-15, 15),
            scale=(0.85, 1.15),
            translate_percent=(-0.1, 0.1),
            shear=(-5, 5),
            p=0.5
        ),
    ], bbox_params=A.BboxParams(
        format='yolo',
        label_fields=['class_ids'],
        min_visibility=0.3,
        min_area=100,
    ))

    total_generated = 0

    for cls_id in sorted(class_counts):
        current = class_counts[cls_id]
        name = class_names[cls_id] if cls_id < len(class_names) else f'class_{cls_id}'
        need = target_count - current

        if need <= 0:
            print(f"\n  [{cls_id}] {name:15s}: {current:5d} — sudah cukup ✅")
            continue

        print(f"\n  [{cls_id}] {name:15s}: {current:5d} → target {target_count} (perlu +{need})")

        source_files = class_files.get(cls_id, [])
        if not source_files:
            print(f"    ⚠️  Tidak ada file sumber untuk kelas {name}!")
            continue

        generated = 0
        attempts = 0
        max_attempts = need * 5  # Batas percobaan

        pbar = tqdm(total=need, desc=f"    Augmenting {name}", ncols=80)

        while generated < need and attempts < max_attempts:
            # Random pilih file sumber
            img_path, lbl_path = random.choice(source_files)
            attempts += 1

            # Baca gambar
            img = cv2.imread(img_path)
            if img is None:
                continue
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            h, w = img.shape[:2]

            # Baca label YOLO
            bboxes = []
            class_ids = []
            with open(lbl_path) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        cid = int(parts[0])
                        cx, cy, bw, bh = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
                        # Clamp values
                        cx = max(0.001, min(0.999, cx))
                        cy = max(0.001, min(0.999, cy))
                        bw = max(0.001, min(0.999, bw))
                        bh = max(0.001, min(0.999, bh))
                        bboxes.append([cx, cy, bw, bh])
                        class_ids.append(cid)

            if not bboxes:
                continue

            # Augmentasi
            try:
                result = augmenter(
                    image=img_rgb,
                    bboxes=bboxes,
                    class_ids=class_ids
                )
            except Exception:
                continue

            aug_bboxes = result['bboxes']
            aug_class_ids = result['class_ids']

            # Cek apakah kelas target masih ada setelah augmentasi
            if cls_id not in aug_class_ids:
                continue

            if len(aug_bboxes) == 0:
                continue

            # Simpan gambar augmentasi
            aug_img = cv2.cvtColor(result['image'], cv2.COLOR_RGB2BGR)
            base_name = os.path.splitext(os.path.basename(img_path))[0]
            new_name = f"{base_name}_aug{cls_id}_{generated}"

            new_img_path = os.path.join(train_img_dir, f"{new_name}.jpg")
            new_lbl_path = os.path.join(train_lbl_dir, f"{new_name}.txt")

            cv2.imwrite(new_img_path, aug_img)

            # Simpan label
            with open(new_lbl_path, 'w') as f:
                for bbox, cid in zip(aug_bboxes, aug_class_ids):
                    cx, cy, bw, bh = bbox
                    f.write(f"{cid} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}\n")

            generated += 1
            total_generated += 1
            pbar.update(1)

        pbar.close()
        print(f"    → Berhasil generate {generated} gambar augmentasi")

    print(f"\n{'='*50}")
    print(f"  ✅ Total gambar augmentasi dibuat: {total_generated}")
    print(f"{'='*50}")

# Jalankan balancing
balance_dataset(DATASET_DIR, class_names)

In [ ]:
# Analisis ulang setelah balancing
balanced_stats, _ = analyze_dataset(DATASET_DIR, "Dataset SETELAH BALANCING")

## 🔧 Langkah 4: Persiapan Training

Fix path di `data.yaml` dan konfigurasi training.

In [ ]:
# Fix data.yaml dengan absolute path
data_yaml_path = os.path.join(DATASET_DIR, 'data.yaml')
with open(data_yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

data_config['train'] = os.path.join(DATASET_DIR, 'train', 'images')
data_config['val'] = os.path.join(DATASET_DIR, 'valid', 'images')
test_path = os.path.join(DATASET_DIR, 'test', 'images')
if os.path.exists(test_path):
    data_config['test'] = test_path

fixed_yaml = os.path.join(WORK_DIR, 'data_fixed.yaml')
with open(fixed_yaml, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)

print(f"✅ data.yaml diperbaiki: {fixed_yaml}")
print(f"\nIsi:")
with open(fixed_yaml) as f:
    print(f.read())

In [ ]:
# ==========================================
# ⚙️ KONFIGURASI TRAINING — UBAH DI SINI!
# ==========================================

# Model base (Level 2A: gunakan model lebih besar)
# Pilihan:
#   "yolo11n.pt"  → Nano    (2.6M params)  — TIDAK DIREKOMENDASIKAN
#   "yolo11s.pt"  → Small   (9M params)    — Cepat, akurasi sedang
#   "yolo11m.pt"  → Medium  (20M params)   — REKOMENDASI (balanced)
#   "yolo11l.pt"  → Large   (43M params)   — Lebih akurat, lebih lambat
#   "yolo11x.pt"  → XLarge  (57M params)   — Paling akurat, paling lambat
#
# Untuk Colab Free (T4 16GB VRAM):
#   yolo11m → batch 16 ✅
#   yolo11l → batch 8  ✅
#   yolo11x → batch 4  ⚠️ (mungkin OOM)

BASE_MODEL = "yolo11m.pt"    # ← UBAH JIKA PERLU
BATCH_SIZE = 16              # ← Kurangi jika CUDA OOM
EPOCHS = 200                 # ← Tambah jika punya waktu
IMAGE_SIZE = 640             # ← Jangan kurangi (penting untuk detail)
PATIENCE = 40                # ← Early stopping

print(f"\n🍈 Konfigurasi Training Level 2C")
print(f"{'='*40}")
print(f"  Model     : {BASE_MODEL}")
print(f"  Batch     : {BATCH_SIZE}")
print(f"  Epochs    : {EPOCHS}")
print(f"  Image Size: {IMAGE_SIZE}")
print(f"  Patience  : {PATIENCE}")

## 🚀 Langkah 5: Training! (Level 2A + 2B + 2C)

Ini adalah cell utama training. Akan memakan waktu **1-3 jam** tergantung GPU dan jumlah epoch.

> ⚠️ **Jangan tutup tab Colab selama training!** Jika perlu, aktifkan "Prevent screen from sleeping".

In [ ]:
from ultralytics import YOLO
import time

# Load base model (auto-download)
print(f"📥 Memuat model dasar: {BASE_MODEL}")
model = YOLO(BASE_MODEL)

print(f"\n🏋️ Training dimulai...")
print(f"   Estimasi waktu: 1-3 jam (tergantung GPU dan epoch)")
print(f"   Ctrl+C untuk menghentikan (model terakhir tetap tersimpan)\n")

start_time = time.time()

results = model.train(
    data=fixed_yaml,
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=0,  # GPU
    patience=PATIENCE,

    # Learning rate
    lr0=0.01,
    lrf=0.01,  # Final LR = lr0 * lrf
    warmup_epochs=5,
    warmup_momentum=0.8,

    # Optimizer
    optimizer='AdamW',
    weight_decay=0.001,
    cos_lr=True,

    # === AUGMENTASI AGRESIF (Level 2C) ===
    hsv_h=0.02,         # Variasi hue (warna)
    hsv_s=0.8,          # Variasi saturasi
    hsv_v=0.5,          # Variasi brightness
    degrees=15.0,        # Rotasi ±15°
    translate=0.15,      # Translasi 15%
    scale=0.6,           # Skala variasi 60%
    shear=5.0,           # Shear ±5°
    perspective=0.001,   # Perspektif
    fliplr=0.5,          # Flip horizontal 50%
    flipud=0.0,          # Tidak flip vertikal
    mosaic=1.0,          # Mosaic ON
    mixup=0.2,           # Mixup 20%
    copy_paste=0.15,     # Copy-paste augmentation
    erasing=0.3,         # Random erasing 30%
    crop_fraction=0.2,   # Random crop
    close_mosaic=15,     # Matikan mosaic 15 epoch terakhir

    # Output
    project=f"{WORK_DIR}/runs",
    name="durian_v2",
    exist_ok=True,
    save=True,
    save_period=50,
    plots=True,
    verbose=True,
)

elapsed = time.time() - start_time
hours = int(elapsed // 3600)
mins = int((elapsed % 3600) // 60)
print(f"\n{'='*50}")
print(f"  ✅ Training selesai dalam {hours}j {mins}m")
print(f"{'='*50}")

## 📊 Langkah 6: Evaluasi Model

Melihat confusion matrix, per-class AP, dan metrik keseluruhan.

In [ ]:
from ultralytics import YOLO
from IPython.display import Image, display

# Path ke best.pt
best_pt = f"{WORK_DIR}/runs/durian_v2/weights/best.pt"

if not os.path.exists(best_pt):
    print(f"❌ best.pt tidak ditemukan di: {best_pt}")
    # Coba cari
    for root, dirs, files in os.walk(f"{WORK_DIR}/runs"):
        if 'best.pt' in files:
            best_pt = os.path.join(root, 'best.pt')
            print(f"   Ditemukan di: {best_pt}")
            break

print(f"\n📊 Mengevaluasi model: {best_pt}")
model = YOLO(best_pt)

# Validasi
metrics = model.val(data=fixed_yaml, plots=True)

print(f"\n{'='*50}")
print(f"  📊 HASIL EVALUASI")
print(f"{'='*50}")
print(f"\n  Metrik Keseluruhan:")
print(f"  ├── mAP50     : {metrics.box.map50:.4f}")
print(f"  ├── mAP50-95  : {metrics.box.map:.4f}")
print(f"  ├── Precision : {metrics.box.mp:.4f}")
print(f"  └── Recall    : {metrics.box.mr:.4f}")

# Per-class metrics
if hasattr(metrics.box, 'ap50') and metrics.box.ap50 is not None:
    print(f"\n  Per-Kelas AP50:")
    names = model.names
    ap50 = metrics.box.ap50
    for i, ap in enumerate(ap50):
        name = names.get(i, f'class_{i}')
        bar = '█' * int(ap * 20)
        status = '✅' if ap > 0.7 else '⚠️' if ap > 0.5 else '❌'
        print(f"    {status} [{i}] {name:15s}: {ap:.4f} {bar}")

In [ ]:
# Tampilkan grafik training
from IPython.display import Image, display
import glob

run_dir = os.path.dirname(os.path.dirname(best_pt))  # runs/durian_v2/

# Training curves
plots = [
    ('results.png', 'Training & Validation Curves'),
    ('confusion_matrix_normalized.png', 'Confusion Matrix (Normalized)'),
    ('confusion_matrix.png', 'Confusion Matrix'),
    ('F1_curve.png', 'F1 Curve'),
    ('PR_curve.png', 'Precision-Recall Curve'),
    ('P_curve.png', 'Precision Curve'),
    ('R_curve.png', 'Recall Curve'),
]

for filename, title in plots:
    path = os.path.join(run_dir, filename)
    if os.path.exists(path):
        print(f"\n📊 {title}")
        display(Image(path, width=800))
    else:
        # Cek di subfolder val
        val_path = os.path.join(run_dir, 'val', filename)
        if os.path.exists(val_path):
            print(f"\n📊 {title}")
            display(Image(val_path, width=800))

## 💾 Langkah 7: Simpan Hasil ke Google Drive & Download

Model `best.pt` yang sudah dilatih akan:
1. Disalin ke Google Drive (aman jika Colab disconnect)
2. Bisa didownload langsung ke laptop

In [ ]:
import shutil
from datetime import datetime

# Buat folder output di Drive
os.makedirs(OUTPUT_DRIVE, exist_ok=True)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# 1. Copy best.pt
best_dst = os.path.join(OUTPUT_DRIVE, 'best.pt')
shutil.copy2(best_pt, best_dst)
print(f"✅ best.pt → {best_dst}")

# 2. Copy dengan timestamp (backup)
backup_dst = os.path.join(OUTPUT_DRIVE, f'best_{timestamp}.pt')
shutil.copy2(best_pt, backup_dst)
print(f"✅ Backup  → {backup_dst}")

# 3. Copy last.pt juga
last_pt = best_pt.replace('best.pt', 'last.pt')
if os.path.exists(last_pt):
    shutil.copy2(last_pt, os.path.join(OUTPUT_DRIVE, 'last.pt'))
    print(f"✅ last.pt → {OUTPUT_DRIVE}/last.pt")

# 4. Copy training plots
run_dir = os.path.dirname(os.path.dirname(best_pt))
plots_dir = os.path.join(OUTPUT_DRIVE, f'plots_{timestamp}')
os.makedirs(plots_dir, exist_ok=True)

for ext in ['*.png', '*.jpg', '*.csv']:
    for f in glob.glob(os.path.join(run_dir, ext)):
        shutil.copy2(f, plots_dir)
    for f in glob.glob(os.path.join(run_dir, 'val', ext)):
        shutil.copy2(f, plots_dir)

print(f"✅ Grafik  → {plots_dir}")

# Ukuran file
size_mb = os.path.getsize(best_pt) / (1024 * 1024)
print(f"\n📦 Ukuran model: {size_mb:.1f} MB")
print(f"\n{'='*50}")
print(f"  Semua file tersimpan di Google Drive:")
print(f"  📁 {OUTPUT_DRIVE}")
print(f"{'='*50}")

In [ ]:
# Download best.pt langsung ke laptop
from google.colab import files

print("📥 Downloading best.pt ke laptop...")
print("   (Browser akan memunculkan dialog download)\n")
files.download(best_pt)

## 🔄 Langkah 8: Cara Menggunakan Model Baru

Setelah `best.pt` terdownload:

1. **Ganti model lama**:
   ```
   Salin best.pt yang baru ke:
   d:\GUI Duren\GUI Duren\best.pt
   (timpa file lama)
   ```

2. **Jalankan GUI Duren**:
   ```bash
   cd "d:\GUI Duren\GUI Duren"
   python main.py
   ```

3. **Selesai!** Model baru otomatis dipakai karena nama file sama (`best.pt`)

### ⚠️ Catatan Penting
- Model baru (yolo11m) berukuran **~40 MB** (vs 5 MB sebelumnya)
- Inference di CPU akan **lebih lambat** (~500-800ms vs ~280ms)
- Jika terlalu lambat, pertimbangkan `yolo11s.pt` sebagai kompromi
- Jika punya GPU NVIDIA di laptop → otomatis pakai CUDA (jauh lebih cepat)

---

### 🎯 Tips Meningkatkan Akurasi Lebih Lanjut
1. **Tambah lebih banyak gambar** terutama untuk kelas minoritas
2. **Jalankan notebook ini lagi** dengan `EPOCHS = 300` dan `BASE_MODEL = "yolo11l.pt"`
3. **Bersihkan dataset**: hapus gambar buram atau label yang salah
4. **Tambah varietas baru**: tambah kelas di data.yaml + label gambar baru

---

## 🔬 (Opsional) Tes Inferensi dengan Gambar

Upload gambar durian untuk tes model baru.

In [ ]:
from google.colab import files
from ultralytics import YOLO
from IPython.display import Image, display
import cv2

print("📷 Upload gambar durian untuk dites:")
uploaded = files.upload()

if uploaded:
    model = YOLO(best_pt)

    for filename, content in uploaded.items():
        # Simpan file
        test_path = f"/content/{filename}"
        with open(test_path, 'wb') as f:
            f.write(content)

        # Prediksi
        results = model.predict(
            test_path,
            conf=0.25,
            imgsz=640,
            save=True,
            project='/content/test_results',
            name='predict',
            exist_ok=True,
        )

        # Tampilkan hasil
        result_img = f"/content/test_results/predict/{filename}"
        if os.path.exists(result_img):
            print(f"\n📊 Hasil deteksi: {filename}")
            display(Image(result_img, width=600))

        # Print deteksi
        for r in results:
            boxes = r.boxes
            if boxes is not None and len(boxes) > 0:
                for i in range(len(boxes)):
                    conf = float(boxes.conf[i])
                    cls = int(boxes.cls[i])
                    name = r.names.get(cls, '?')
                    print(f"  🍈 {name} — {conf*100:.1f}%")
            else:
                print("  Tidak ada durian terdeteksi")
else:
    print("Tidak ada gambar diupload")